# Algoritmo Apriori

Notebook elaborado siguiendo la rúbrica de la asignatura (ver `rubrica.md`).

**Autor:** _completar_  
**Fecha:** _completar_

## 1. Descripción

**Apriori** es un algoritmo clásico de **minería de reglas de asociación** y descubrimiento de **itemsets frecuentes** en bases de datos transaccionales. Fue propuesto por Rakesh Agrawal y Ramakrishnan Srikant en 1994 en IBM Almaden Research Center.

Su objetivo es encontrar relaciones del tipo:

> *Si un cliente compra pan y mantequilla, entonces es probable que también compre leche.*

Formalmente, dada una base de transacciones $D = \{T_1, T_2, \dots, T_n\}$ donde cada $T_i$ es un conjunto de ítems, Apriori busca todos los itemsets $X \subseteq I$ cuyo **soporte** supere un umbral mínimo, y a partir de ellos genera **reglas de asociación** $X \Rightarrow Y$ con métricas de calidad (confianza, lift, etc.).

Se basa en el **Principio Apriori**:

> *Si un itemset es frecuente, todos sus subconjuntos también lo son. Equivalentemente, si un itemset es infrecuente, todos sus supraconjuntos también lo serán.*

Esta propiedad permite **podar** drásticamente el espacio de búsqueda.

### Casos de uso típicos
- Análisis de la cesta de la compra (*market basket analysis*).
- Recomendación de productos.
- Detección de patrones en logs web.
- Bioinformática (co-ocurrencia de genes/proteínas).

## 2. Bibtex y Referencias

### BibTeX
```bibtex
@inproceedings{agrawal1994fast,
  title     = {Fast algorithms for mining association rules},
  author    = {Agrawal, Rakesh and Srikant, Ramakrishnan},
  booktitle = {Proc. 20th Int. Conf. Very Large Data Bases, VLDB},
  volume    = {1215},
  pages     = {487--499},
  year      = {1994}
}

@inproceedings{agrawal1993mining,
  title     = {Mining association rules between sets of items in large databases},
  author    = {Agrawal, Rakesh and Imieli{\'n}ski, Tomasz and Swami, Arun},
  booktitle = {Proceedings of the 1993 ACM SIGMOD international conference on Management of data},
  pages     = {207--216},
  year      = {1993}
}
```

### APA
- Agrawal, R., & Srikant, R. (1994). *Fast algorithms for mining association rules*. In **Proc. 20th Int. Conf. Very Large Data Bases, VLDB** (Vol. 1215, pp. 487–499).
- Agrawal, R., Imieliński, T., & Swami, A. (1993). *Mining association rules between sets of items in large databases*. In **Proceedings of the 1993 ACM SIGMOD International Conference on Management of Data** (pp. 207–216).

## 3. Tipo de Modelo

| Criterio | Clasificación |
|---|---|
| **Método de aprendizaje** | No supervisado |
| **Por parámetros** | No paramétrico |
| **Datos de aprendizaje** | Offline (batch) — requiere recorrer la base completa |
| **Resultado del entrenamiento** | Conjunto de **reglas de asociación** + itemsets frecuentes |

Notas:
- **No supervisado** porque no necesita etiquetas; descubre patrones en los datos.
- **No paramétrico** porque no asume una forma funcional ni un número fijo de parámetros a ajustar.
- **Offline** porque la versión clásica necesita múltiples pasadas sobre el dataset (existen variantes online/streaming).

## 4. Algoritmo de Entrenamiento

Apriori es iterativo y por niveles (*level-wise*). Cada nivel $k$ genera itemsets de tamaño $k$ a partir de los frecuentes de tamaño $k-1$.

### Pseudocódigo
```text
Entrada:  D (transacciones), min_sup (soporte mínimo)
Salida:   L = unión de itemsets frecuentes de todos los tamaños

L1 = { itemsets de tamaño 1 con soporte >= min_sup }
k  = 2
mientras L_{k-1} no esté vacío:
    Ck = generar_candidatos(L_{k-1})          # join + prune (Apriori principle)
    para cada transacción t en D:
        para cada c en Ck contenido en t:
            c.contador += 1
    Lk = { c en Ck | soporte(c) >= min_sup }
    k  = k + 1
devolver  L = L1 ∪ L2 ∪ … ∪ L_{k-1}

# Luego, a partir de L se generan reglas X => Y con
# confianza >= min_conf  (y opcionalmente lift > 1).
```

### Métricas clave

Sea $X, Y \subseteq I$ con $X \cap Y = \emptyset$:

- **Soporte:**  $\;\;\text{sup}(X) = \dfrac{|\{T \in D : X \subseteq T\}|}{|D|}$
- **Confianza:** $\;\;\text{conf}(X \Rightarrow Y) = \dfrac{\text{sup}(X \cup Y)}{\text{sup}(X)}$
- **Lift:** $\;\;\text{lift}(X \Rightarrow Y) = \dfrac{\text{conf}(X \Rightarrow Y)}{\text{sup}(Y)}$

Interpretación del lift:
- $\text{lift} = 1$ → $X$ e $Y$ son independientes.
- $\text{lift} > 1$ → asociación positiva.
- $\text{lift} < 1$ → asociación negativa.

## 5. Supuestos y Restricciones

- **Datos transaccionales categóricos:** los ítems deben ser discretos. Variables numéricas requieren discretización previa.
- **Representación binaria:** cada transacción se modela por la *presencia/ausencia* del ítem (no usa cantidades).
- **Umbrales definidos por el usuario:** `min_support` y `min_confidence` son hiperparámetros que condicionan los resultados.
- **Principio Apriori:** la propiedad de antimonotonicidad del soporte es fundamental para podar el espacio de búsqueda.
- **Coste computacional:** crece exponencialmente con el número de ítems distintos; en datasets densos puede ser inviable.
- **Múltiples pasadas sobre los datos:** requiere $k$ recorridos de la base, lo que penaliza el I/O.
- **Reglas redundantes:** suele generar muchas reglas similares; conviene un post-filtrado (cierre, máximos, lift, etc.).
- **No considera el orden ni el tiempo** entre transacciones (para eso existen variantes como GSP o PrefixSpan).

## 6. Tests / Métricas de validación

Apriori no se valida con tests estadísticos clásicos como una regresión, sino con **métricas de interés** sobre las reglas descubiertas:

- **Soporte (support)** — frecuencia relativa del itemset.
- **Confianza (confidence)** — probabilidad condicional $P(Y \mid X)$.
- **Lift** — cuánto se desvía la regla de la independencia estadística.
- **Leverage** — diferencia entre la frecuencia observada y la esperada bajo independencia.
- **Conviction** — fortaleza de la implicación; $\infty$ si la regla es perfecta.

Para evaluar la utilidad práctica suele complementarse con análisis cualitativo (¿la regla tiene sentido para el dominio?) y validación con datos nuevos.

---
## 7. Implementación práctica

Usaremos la librería **`mlxtend`** (Machine Learning Extensions) de Sebastian Raschka, que implementa Apriori y reglas de asociación de forma idiomática.

### 7.1 Instalación e imports

In [ ]:
# Si se ejecuta en Colab o un entorno sin mlxtend, descomentar:
# !pip install mlxtend pandas

import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

print('mlxtend listo')

### 7.2 Dataset de ejemplo — Cesta de la compra

Simulamos una pequeña base de 10 transacciones de un supermercado.

In [ ]:
transacciones = [
    ['pan', 'leche', 'huevos'],
    ['pan', 'pañales', 'cerveza', 'huevos'],
    ['leche', 'pañales', 'cerveza', 'cola'],
    ['pan', 'leche', 'pañales', 'cerveza'],
    ['pan', 'leche', 'pañales', 'cola'],
    ['pan', 'leche'],
    ['pan', 'cerveza'],
    ['leche', 'pañales'],
    ['pan', 'leche', 'pañales'],
    ['cerveza', 'cola'],
]

print(f'Número de transacciones: {len(transacciones)}')
for i, t in enumerate(transacciones, 1):
    print(f'T{i:>2}: {t}')

### 7.3 Codificación one-hot

`mlxtend` necesita un DataFrame booleano donde cada columna es un ítem y cada fila una transacción.

In [ ]:
te = TransactionEncoder()
te_array = te.fit(transacciones).transform(transacciones)
df = pd.DataFrame(te_array, columns=te.columns_)
df

### 7.4 Itemsets frecuentes con Apriori

Buscamos itemsets cuyo soporte sea >= 0.3 (es decir, presentes en al menos 3 de las 10 transacciones).

In [ ]:
min_sup = 0.3
frecuentes = apriori(df, min_support=min_sup, use_colnames=True)
frecuentes = frecuentes.sort_values('support', ascending=False).reset_index(drop=True)
frecuentes

### 7.5 Generación de reglas de asociación

A partir de los itemsets frecuentes, generamos reglas con **confianza >= 0.6**.

In [ ]:
reglas = association_rules(frecuentes, metric='confidence', min_threshold=0.6)
columnas = ['antecedents', 'consequents', 'support', 'confidence', 'lift', 'leverage', 'conviction']
reglas = reglas[columnas].sort_values('lift', ascending=False).reset_index(drop=True)
reglas

### 7.6 Interpretación

- Filas con **lift > 1** indican que el antecedente y el consecuente aparecen juntos *más* de lo que se esperaría por azar.
- La regla clásica **{pañales} ⇒ {cerveza}** (o variantes) ilustra el famoso caso de estudio del *market basket analysis*.
- Ajustando `min_support` y `min_threshold` se controla el balance entre **número de reglas** y **fuerza** de las mismas:
  - Umbrales altos → pocas reglas, muy confiables.
  - Umbrales bajos → muchas reglas, posiblemente ruidosas.

In [ ]:
# Top 5 reglas por lift
print('Top 5 reglas por LIFT:')
for _, r in reglas.head(5).iterrows():
    ant = ', '.join(sorted(r['antecedents']))
    con = ', '.join(sorted(r['consequents']))
    print(f'  {{{ant}}}  =>  {{{con}}}'
          f'   sup={r["support"]:.2f}  conf={r["confidence"]:.2f}  lift={r["lift"]:.2f}')

## 8. Conclusión

- Apriori es un algoritmo **no supervisado y no paramétrico** que descubre **reglas de asociación** en datos transaccionales.
- Su elegancia está en el **principio de antimonotonicidad** del soporte, que poda el espacio de búsqueda.
- Limitación principal: el coste crece con el número de ítems distintos; en datasets grandes o densos se prefieren variantes como **FP-Growth** o **ECLAT**.
- En la práctica, la calidad de las reglas depende fuertemente de la elección de los umbrales `min_support` y `min_confidence`, y del **post-filtrado** mediante lift, conviction, etc.